# Notebook 1 — Offline RL: Building a Simulator from Data

**Module 3 — Applied Tabular RL**

## What this notebook covers
1. Generating an offline dataset from a known environment
2. Fitting a demand model (world model) from that dataset
3. Training Q-learning on the fitted simulator
4. Diagnosing compounding rollout error

## Environment: Toy Market MDP
| | |
|---|---|
| **States** | Demand temperature: 0 (cold) → 4 (hot) |
| **Actions** | Price multiplier: 0.85× (discount), 1.00× (standard), 1.15× (premium) |
| **Reward** | Revenue = price × quantity |
| **True dynamics** | Quantity responds to price with state-dependent elasticity |

The true dynamics are **unknown** to the agent — it sees only logged data.

---
> **M2 callback:** In Module 2 we ran Q-learning on FrozenLake where the transition function was given. Here we must *estimate* the transition function from a fixed offline dataset before training can begin.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

SEED = 42

## Part 1 — The True Environment

We define a toy market where demand responds to price with **state-dependent elasticity**:

$$\hat{q}(s, p) = b_s \cdot \left(\frac{p}{p_0}\right)^{e_s}$$

- $b_s$: base demand in state $s$ (units per period)
- $p_0 = 10.0$: reference price
- $e_s < 0$: price elasticity in state $s$ (more negative = more elastic)

Revenue reward: $r = p \cdot \hat{q}(s, p)$

This is the **hidden ground truth** — the agent never has direct access to $b_s$ or $e_s$.

In [ ]:
BASE_PRICE   = 10.0
ACTION_MULTS = np.array([0.85, 1.00, 1.15])
N_ACTIONS    = len(ACTION_MULTS)
N_STATES     = 5

# True parameters (hidden from the agent)
TRUE_BASE_DEMAND = np.array([3.0,  8.0,  15.0, 25.0, 40.0])
TRUE_ELASTICITY  = np.array([-2.0, -1.5, -1.0, -0.7, -0.3])
A_NAMES          = ['discount', 'standard', 'premium']

_BINS = [5, 12, 20, 32]  # quantity -> state bin thresholds

def qty_bin(q: float) -> int:
    for i, t in enumerate(_BINS):
        if q < t:
            return i
    return len(_BINS)

def true_demand(s: int, mult: float) -> float:
    return float(TRUE_BASE_DEMAND[s] * (mult ** TRUE_ELASTICITY[s]))

def true_step(s: int, a: int) -> tuple[int, float]:
    mult = float(ACTION_MULTS[a])
    qty  = true_demand(s, mult)
    return qty_bin(qty), BASE_PRICE * mult * qty

# Display true revenues and optimal policy
print(f"{'s':>3}  {'e_s':>5}  {'disc rev':>10}  {'std rev':>9}  {'prem rev':>10}  {'optimal':>10}")
for s in range(N_STATES):
    revs = [BASE_PRICE * ACTION_MULTS[a] * true_demand(s, ACTION_MULTS[a]) for a in range(N_ACTIONS)]
    opt  = A_NAMES[int(np.argmax(revs))]
    print(f"{s:>3}  {TRUE_ELASTICITY[s]:>5.1f}  {revs[0]:>10.2f}  {revs[1]:>9.2f}  {revs[2]:>10.2f}  {opt:>10}")

## Part 2 — Generating the Offline Dataset

In a real offline RL setting we inherit a fixed dataset $\mathcal{D}$ — logged decisions and outcomes from a previous **behavior policy** $\pi_b$.

Here we simulate that: a uniform random policy takes 1,000 actions across random states, and we record the noisy outcomes.

$$\mathcal{D} = \{(s_i,\, a_i,\, q_i)\}_{i=1}^{N}, \quad q_i = \hat{q}(s_i, a_i) \cdot e^{\epsilon_i}, \quad \epsilon_i \sim \mathcal{N}(0, \sigma^2)$$

**Key constraint:** no additional experiments are allowed. Everything the agent learns comes from $\mathcal{D}$.

In [ ]:
def generate_dataset(n: int = 1000, noise_std: float = 0.25, seed: int = SEED):
    rng     = np.random.default_rng(seed)
    states  = rng.integers(0, N_STATES, size=n)
    actions = rng.integers(0, N_ACTIONS, size=n)   # uniform random behavior policy
    noise   = rng.normal(0.0, noise_std, size=n)
    qtys    = np.array([
        max(0.0, true_demand(int(states[i]), ACTION_MULTS[int(actions[i])]) * np.exp(noise[i]))
        for i in range(n)
    ])
    return states, actions, qtys

D_states, D_actions, D_qtys = generate_dataset()
D_prices = BASE_PRICE * ACTION_MULTS[D_actions]

print(f"Dataset: {len(D_states):,} transitions | behavior policy: uniform random")
print(f"State distribution:  {np.round(np.bincount(D_states) / len(D_states), 3)}")
print(f"Action distribution: {np.round(np.bincount(D_actions) / len(D_actions), 3)}")
print(f"Qty  —  min: {D_qtys.min():.1f}  median: {np.median(D_qtys):.1f}  max: {D_qtys.max():.1f}")

## Part 3 — Fitting the Demand Model

We fit a **log-linear demand model** separately for each state:

$$\log(1 + \hat{q}) = \alpha_s + \beta_s \cdot \log(p)$$

The estimated coefficient $\hat{\beta}_s$ is our approximation of the true price elasticity $e_s$.

Fitting is done via OLS on the logged observations in $\mathcal{D}$ restricted to state $s$.

> Note how many samples fall in each state — states with few observations yield less reliable elasticity estimates.

In [ ]:
state_models: dict[int, LinearRegression] = {}

print(f"{'s':>3}  {'n':>5}  {'fitted beta':>12}  {'true e_s':>10}  {'error':>8}")
for s in range(N_STATES):
    mask = D_states == s
    n    = int(mask.sum())
    if n < 5:
        print(f"{s:>3}  {n:>5}  (too few observations)")
        continue
    X = np.log(D_prices[mask]).reshape(-1, 1)
    y = np.log1p(D_qtys[mask])
    m = LinearRegression().fit(X, y)
    state_models[s] = m
    err = m.coef_[0] - TRUE_ELASTICITY[s]
    print(f"{s:>3}  {n:>5}  {m.coef_[0]:>12.3f}  {TRUE_ELASTICITY[s]:>10.1f}  {err:>+8.3f}")

In [ ]:
def sim_demand(s: int, mult: float) -> float:
    """Predicted demand from the fitted log-linear model."""
    m = state_models.get(s)
    if m is None:                     # fallback for states with no logged data
        return true_demand(s, mult)
    log_q = float(m.predict([[np.log(BASE_PRICE * mult)]])[0])
    return max(0.0, float(np.expm1(np.clip(log_q, -10.0, 10.0))))

def sim_step(s: int, a: int) -> tuple[int, float]:
    mult = float(ACTION_MULTS[a])
    qty  = sim_demand(s, mult)
    return qty_bin(qty), BASE_PRICE * mult * qty

# Sanity check: simulator vs true revenue
print(f"{'s':>3}  {'a':>3}  {'true_rev':>10}  {'sim_rev':>10}  {'pct_err':>9}")
for s in range(N_STATES):
    for a in range(N_ACTIONS):
        _, tr = true_step(s, a)
        _, sr = sim_step(s, a)
        print(f"{s:>3}  {a:>3}  {tr:>10.2f}  {sr:>10.2f}  {100.0*(sr - tr)/(tr + 1e-9):>+8.1f}%")

## Part 4 — Q-Learning on the Fitted Simulator

We train a tabular Q-policy on the simulator. The agent **never touches the true environment during training**.

**Q-table shape:** $(5\ \text{states} \times 3\ \text{actions})$ = 15 parameters.

Q-learning update (off-policy, bootstrapped):

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big(r + \gamma \max_{a'} Q(s', a') - Q(s, a)\Big)$$

After training we compare the learned policy against the true analytical optimum.

In [ ]:
def train_q(
    step_fn,
    n_states:  int   = N_STATES,
    n_actions: int   = N_ACTIONS,
    n_ep:      int   = 4000,
    horizon:   int   = 8,
    alpha:     float = 0.10,
    gamma:     float = 0.95,
    eps_start: float = 1.0,
    eps_end:   float = 0.05,
    seed:      int   = 0,
) -> tuple[np.ndarray, list[float]]:
    rng  = np.random.default_rng(seed)
    Q    = np.zeros((n_states, n_actions))
    rets = []
    for ep in range(n_ep):
        eps = eps_end + (eps_start - eps_end) * np.exp(-5.0 * ep / n_ep)
        s   = int(rng.integers(0, n_states))
        G   = 0.0
        for _ in range(horizon):
            a     = int(rng.integers(0, n_actions)) if rng.random() < eps else int(np.argmax(Q[s]))
            s2, r = step_fn(s, a)
            Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) - Q[s, a])
            G += r
            s  = s2
        rets.append(G)
    return Q, rets

In [ ]:
Q_sim, sim_rets = train_q(sim_step, seed=SEED)

# Learning curve
W      = 200
smooth = np.convolve(sim_rets, np.ones(W) / W, mode='valid')
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(smooth)
ax.set_xlabel("Episode")
ax.set_ylabel(f"Return ({W}-ep moving avg)")
ax.set_title("Q-Learning on Fitted Simulator")
plt.tight_layout()
plt.show()

# True analytical optimum
TRUE_OPT = [
    int(np.argmax([BASE_PRICE * ACTION_MULTS[a] * true_demand(s, ACTION_MULTS[a])
                   for a in range(N_ACTIONS)]))
    for s in range(N_STATES)
]
learned = np.argmax(Q_sim, axis=1)

print(f"\n{'s':>3}  {'learned':>12}  {'true opt':>12}  match")
for s in range(N_STATES):
    chk = 'OK' if learned[s] == TRUE_OPT[s] else '--'
    print(f"{s:>3}  {A_NAMES[learned[s]]:>12}  {A_NAMES[TRUE_OPT[s]]:>12}  {chk}")

## Part 5 — Compounding Error in Multi-Step Rollouts

Each simulator step uses the *predicted* quantity $\hat{q}_t$ to build the next state, rather than the real outcome. Over $T$ steps, small per-step errors accumulate:

$$\hat{s}_t = f(\hat{s}_{t-1}, a_{t-1};\,\theta) + \text{error}_{t-1}$$

After $T$ steps the simulator's **state distribution** can diverge substantially from what the true environment would produce.

We measure this with **total variation (TV) distance** between the two state distributions at each step:

$$\text{TV}(t) = \frac{1}{2} \sum_{s} \left| P_{\text{true}}(s_t = s) - P_{\text{sim}}(s_t = s) \right|$$

In [ ]:
def rollout(
    policy:   np.ndarray,
    step_fn,
    horizon:  int,
    n:        int,
    seed:     int = 0,
) -> np.ndarray:
    """Returns (n, horizon) array of visited states."""
    rng    = np.random.default_rng(seed)
    states = np.zeros((n, horizon), dtype=int)
    for i in range(n):
        s = int(rng.integers(0, N_STATES))
        for t in range(horizon):
            states[i, t] = s
            s, _         = step_fn(s, int(policy[s]))
    return states

policy    = np.argmax(Q_sim, axis=1)
HOR, NROL = 20, 600

true_traj = rollout(policy, true_step, HOR, NROL, seed=1)
sim_traj  = rollout(policy, sim_step,  HOR, NROL, seed=1)

# TV distance at each step
tv = np.array([
    0.5 * np.sum(np.abs(
        np.bincount(true_traj[:, t], minlength=N_STATES) / NROL -
        np.bincount(sim_traj[:,  t], minlength=N_STATES) / NROL
    ))
    for t in range(HOR)
])

# Heatmaps of state distribution
state_labels = ['cold', 'low', 'normal', 'warm', 'hot']
fig, axes    = plt.subplots(1, 2, figsize=(11, 4))
for ax, traj, title in zip(axes, [true_traj, sim_traj], ['True environment', 'Simulator']):
    dist = np.stack([
        np.bincount(traj[:, t], minlength=N_STATES) / NROL
        for t in range(HOR)
    ])
    im = ax.imshow(dist.T, aspect='auto', origin='lower', cmap='Blues', vmin=0, vmax=0.8)
    ax.set_xlabel("Step")
    ax.set_ylabel("State")
    ax.set_title(title)
    ax.set_yticks(range(N_STATES))
    ax.set_yticklabels(state_labels)
plt.colorbar(im, ax=axes.ravel().tolist(), label='Fraction of rollouts')
plt.suptitle("State Distribution Over Rollout Horizon", y=1.01)
plt.tight_layout()
plt.show()

# TV distance over time
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, HOR + 1), tv, marker='o', markersize=4)
ax.set_xlabel("Step")
ax.set_ylabel("TV distance")
ax.set_title("Divergence: True vs Simulator State Distributions")
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()

print(f"TV distance  |  step 1: {tv[0]:.3f}   step 5: {tv[4]:.3f}   step 20: {tv[19]:.3f}")

## Summary

| Step | What we did |
|------|-------------|
| True environment | Defined a toy MDP with state-dependent price elasticity — unknown to the agent |
| Offline dataset | Logged 1,000 noisy transitions from a uniform random behavior policy |
| Demand model | Fitted per-state log-linear model from logged data; recovered elasticities approximately |
| Simulator | Substituted the fitted model as the environment transition function |
| Q-learning | Trained tabular policy on the simulator; learned the correct state-dependent strategy |
| Compounding error | Simulator and true state distributions diverge over longer horizons |

**Key takeaway:** A well-fitted simulator makes offline RL viable for short horizons. Compounding error limits multi-step reliability — the shallower the model, the shorter the safe planning horizon.

---
**Next:** Notebook 2 — Reward Design and Policy Evaluation: what happens when the reward function is wrong, and how to rigorously test whether a policy is actually better.